# Homework 7.1 - Part I

## Define the Spaces

### Conditioned Generation: From Parameters to Dynamics to Images

The scientific generation pipeline is

$$
\boxed{
c
\longmapsto
\gamma_c
\longmapsto
I_c
}
$$

where

- $c$ is the conditioning information;
- $\gamma_c$ is the Lorenz trajectory determined by that condition;
- $I_c$ is a visual representation of the trajectory.

The key point is that the image is **not** the primary mathematical object.

The condition first determines a dynamical system and its trajectory. Only afterward is that trajectory rendered as an image.

# 1. Condition Space

For the Lorenz system, the condition vector is

$$
c
=
(\sigma,\rho,\beta,x_0,y_0,z_0,T).
$$

Therefore

$$
c\in\mathbb{R}^7.
$$

Given $$c$$, solve

$$
\dot{x}
=
\sigma(y-x),
$$

$$
\dot{y}
=
x(\rho-z)-y,
$$

$$
\dot{z}
=
xy-\beta z,
$$

with initial condition

$$
(x(0),y(0),z(0))
=
(x_0,y_0,z_0).
$$

Thus the first deterministic map is

$$
F:
c
\longmapsto
\gamma_c,
$$

where

$$
\gamma_c(t)
=
(x(t),y(t),z(t)),
\qquad
0\le t\le T.
$$

## Reference condition

For Parts II and III, the primary reference condition is the classical chaotic Lorenz configuration

$$
c_\star
=
\left(
10,
28,
\frac{8}{3},
1,
1,
1,
25
\right).
$$

Thus

$$
\sigma=10,
\qquad
\rho=28,
\qquad
\beta=\frac{8}{3},
$$

with

$$
(x_0,y_0,z_0)=(1,1,1),
\qquad
T=25.
$$

This reference case will be used to verify the numerical simulation and the trajectory-to-image rendering map.

# 2. Continuous Trajectory Space

For a fixed condition $$c$$, the exact trajectory is a function

$$
\gamma_c:
[0,T]
\rightarrow
\mathbb{R}^3.
$$

In continuous form,

$$
\gamma_c
\in
C([0,T],\mathbb{R}^3)
$$

for the smooth Lorenz vector field.

This function-space representation is mathematically natural but is not directly stored by a computer.

A numerical simulation instead samples the trajectory at finitely many times.

# 3. Discretized Trajectory Space

Choose

$$
0=t_0<t_1<\cdots<t_M=T.
$$

The sampled trajectory is

$$
\Gamma_c
=
\left(
x(t_0),y(t_0),z(t_0),
\ldots,
x(t_M),y(t_M),z(t_M)
\right).
$$

Therefore

$$
\boxed{
\Gamma_c
\in
\mathbb{R}^{3(M+1)}
}.
$$

For the numerical experiments in Parts II-IV, we use

$$
\Delta t=0.01,
\qquad
T=25.
$$

Hence

$$
M
=
\frac{25}{0.01}
=
2500,
$$

so

$$
M+1=2501
$$

sample times and

$$
\Gamma_c
\in
\mathbb{R}^{7503}.
$$

This fixed sampling convention makes every trajectory in the conditioned dataset have the same numerical shape.

# 4. Image Space

The main machine-readable rendering in Parts III-IV will be a deterministic RGB rendering of the trajectory's

$$
(x,z)
$$

phase-space projection.

Let the rendered image have

$$
H=W=256.
$$

Then each image contains

$$
3HW
=
3(256)(256)
=
196{,}608
$$

RGB channel values.

After normalizing pixel values to the interval $$[0,1]$$, the image may be regarded as a vector

$$
\boxed{
I_c
\in
[0,1]^{196608}
\subset
\mathbb{R}^{196608}
}.
$$

Thus the representation chain is

$$
\boxed{
c\in\mathbb{R}^7
\longmapsto
\Gamma_c\in\mathbb{R}^{7503}
\longmapsto
I_c\in\mathbb{R}^{196608}
}.
$$

# 5. The Two Maps

Define the numerical trajectory map

$$
S:
\mathcal{C}
\rightarrow
\mathbb{R}^{3(M+1)}
$$

by

$$
S(c)=\Gamma_c.
$$

Then define a rendering map

$$
R:
\mathbb{R}^{3(M+1)}
\rightarrow
\mathbb{R}^{3HW}
$$

by

$$
R(\Gamma_c)=I_c.
$$

The full conditioned-generation pipeline is the composition

$$
\boxed{
R\circ S:
c
\longmapsto
\Gamma_c
\longmapsto
I_c
}.
$$

This makes explicit that the image depends on the condition through the mathematical trajectory.

# 6. Parameter Domain for the Conditioned Dataset

Part IV requires us to vary

$$
\sigma,
\qquad
\rho,
\qquad
\beta,
\qquad
(x_0,y_0,z_0).
$$

For the toy dataset, the following ranges are fixed **before generating any trajectories**:

$$
\sigma\in[8,12],
$$

$$
\rho\in[24,32],
$$

$$
\beta\in[2.2,3.0],
$$

$$
x_0,y_0\in[-15,15],
$$

$$
z_0\in[5,35].
$$

We keep

$$
T=25
$$

fixed so all trajectories share the same dimension.

Therefore the dataset condition domain is

$$
\mathcal{C}
=
[8,12]
\times
[24,32]
\times
[2.2,3.0]
\times
[-15,15]
\times
[-15,15]
\times
[5,35]
\times
\{25\}.
$$

Although $$T$$ is fixed in this particular dataset, it remains part of the seven-dimensional condition vector.

In [1]:
from pathlib import Path
import json

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw7_1"
RESULT_DIR = PROJECT_ROOT / "result images" / "hw7_1"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

configuration = {
    "homework": "7.1",
    "condition_coordinates": [
        "sigma",
        "rho",
        "beta",
        "x0",
        "y0",
        "z0",
        "T",
    ],
    "reference_condition": {
        "sigma": 10.0,
        "rho": 28.0,
        "beta": 8.0 / 3.0,
        "x0": 1.0,
        "y0": 1.0,
        "z0": 1.0,
        "T": 25.0,
    },
    "time_discretization": {
        "dt": 0.01,
        "T": 25.0,
        "M": 2500,
        "samples_M_plus_1": 2501,
        "trajectory_vector_dimension": 7503,
    },
    "rendering": {
        "projection": "x-z",
        "height": 256,
        "width": 256,
        "channels": 3,
        "flattened_image_dimension": 196608,
    },
    "dataset_parameter_domain": {
        "sigma": [8.0, 12.0],
        "rho": [24.0, 32.0],
        "beta": [2.2, 3.0],
        "x0": [-15.0, 15.0],
        "y0": [-15.0, 15.0],
        "z0": [5.0, 35.0],
        "T": [25.0, 25.0],
    },
}

path = DATA_DIR / "part1_pipeline_configuration.json"

path.write_text(
    json.dumps(
        configuration,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:", path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.1\data\hw7_1\part1_pipeline_configuration.json


# Part I Conclusion

The project contains three distinct representations:

$$
c
\in
\mathbb{R}^7,
$$

$$
\Gamma_c
\in
\mathbb{R}^{3(M+1)},
$$

and

$$
I_c
\in
\mathbb{R}^{3HW}.
$$

For the fixed numerical choices

$$
T=25,
\qquad
\Delta t=0.01,
\qquad
H=W=256,
$$

the concrete dimensions are

$$
\boxed{
\mathbb{R}^7
\longmapsto
\mathbb{R}^{7503}
\longmapsto
\mathbb{R}^{196608}.
}
$$

Part II will construct the first map numerically and test whether the computed trajectory satisfies the Lorenz differential equations to reasonable accuracy.